# Experiment 1B - Full Training on AWS

**Instance:** ml.g4dn.xlarge (NVIDIA T4 GPU)  
**Duration:** 4-6 hours per variation  
**Total Cost:** ~$3 per variation (~$25 for all 8)  
**Branch:** simran-unified-experiments

---

## What This Trains:

✅ **8 Model Variations:**
- 4 variable sets × 2 learning rates = 8 models

✅ **Data:**
- Training: 231 deep snow HUCs
- Validation: 77 HUCs (for model selection)
- Test Set A: 78 random HUCs
- Test Set B: 81 Yakima/Naches HUCs

✅ **Training:**
- 30 epochs per variation
- GPU training (3-5x faster than CPU)
- Automatic checkpointing every 10 epochs
- MLflow experiment tracking

---

## 🚀 Can You Close Your Laptop?

**YES!** Training runs on AWS cloud server:
- Start training → Close laptop → Go do other things
- Reconnect anytime to check progress
- Training continues 24/7 in the cloud

---

## ⏱️ Timeline:

1. **Setup (10 min):** Install packages, load data
2. **Training (4-6 hrs):** Per variation, runs automatically
3. **Checkpoint (auto):** Saves model every 10 epochs
4. **Results:** MLflow tracks everything automatically

**You only need to:**
- Run setup cells once
- Start training
- Come back later to check results

---

## Variable Sets:

1. **Base:** `[mean_pr, mean_tair, mean_swe_lag_30]`
2. **Base + Wind:** `[mean_pr, mean_tair, mean_swe_lag_30, mean_vs]`
3. **Base + Solar:** `[mean_pr, mean_tair, mean_swe_lag_30, mean_srad]`
4. **Base + Solar + Wind:** `[mean_pr, mean_tair, mean_swe_lag_30, mean_srad, mean_vs]`

**Learning Rates:** 0.001, 0.0003

---

## 💾 Files Needed (Upload to SageMaker):

Before starting, upload these files:
1. `snowml-package-gpu-fixed.tar.gz` (SnowML code)
2. `exp1b_train_hucs.txt` (231 training HUCs)
3. `exp1b_validation_hucs.txt` (77 validation HUCs)

---

Let's begin! 🚀

---
# Step 1: Check GPU (1 minute)
Verify you're on a GPU instance

In [ ]:
import torch

print("="*60)
print("GPU CHECK")
print("="*60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✅ GPU READY - Training will be FAST!")
else:
    print("\n⚠️  No GPU detected - Training will be SLOW!")
    print("    Make sure you launched ml.g4dn.xlarge instance")

---
# Step 2: Install Dependencies (3 minutes)

⚠️ **IMPORTANT:** This installs ALL required packages including xarray/geopandas

You'll see some warnings - **this is normal and OK!**

In [ ]:
print("Installing required packages...")
print("This takes ~3 minutes. You'll see some warnings - that's OK!\n")

# Core packages
!pip install -q mlflow boto3

# Data handling packages (COMPLETE installation for training)
!pip install -q netCDF4 h5py s3fs xarray geopandas

# Upgrade numpy for compatibility
!pip install --upgrade numpy

print("\n✅ All dependencies installed!")
print("Note: Version warnings are expected but won't affect training")

---
# Step 3: Install SnowML Package (30 seconds)

⚠️ **CRITICAL:** After running this cell:
1. Go to: **Kernel → Restart Kernel**
2. Then continue from Step 4

**DO NOT skip the kernel restart!**

In [ ]:
# Extract and install SnowML package
print("Extracting SnowML package...")
!tar -xzf snowml-package-gpu-fixed.tar.gz

print("Installing SnowML with GPU fixes...")
!pip install -q -e .

print("\n✅ SnowML installed!")
print("\n⚠️⚠️⚠️  NOW RESTART THE KERNEL  ⚠️⚠️⚠️")
print("    Kernel → Restart Kernel")
print("    Then continue from Step 4")

---
---
# ⚠️ RESTART KERNEL ABOVE, THEN CONTINUE HERE ⚠️
---
---

# Step 4: Import Libraries (30 seconds)

In [ ]:
from datetime import datetime
import pandas as pd
import torch
from torch import optim
import mlflow
import numpy as np

# Import SnowML modules
from snowML.LSTM import LSTM_pre_process as LSTM_pp
from snowML.LSTM import LSTM_train as LSTM_tr
from snowML.LSTM import LSTM_model as LSTM_mod
from snowML.datapipe.utils import set_data_constants as sdc

print("✅ All libraries imported!")
print(f"   PyTorch: {torch.__version__}")
print(f"   MLflow: {mlflow.__version__}")

---
# Step 5: Configuration (Setup Device & Parameters)

In [ ]:
# Device setup - USE GPU!
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("🚀 GPU training enabled!")
else:
    print("⚠️  GPU not available, using CPU (will be SLOW)")

# Training parameters
n_epochs = 30  # Full training!
hidden_size = 64
num_layers = 1
dropout = 0.3
batch_size = 32
lookback = 180

print(f"\nTraining config:")
print(f"   Epochs: {n_epochs}")
print(f"   Hidden size: {hidden_size}")
print(f"   Batch size: {batch_size}")
print(f"   Lookback: {lookback} days")

---
# Step 6: Define 8 Model Variations

We'll train all 8 combinations of:
- 4 variable sets (Base, +Wind, +Solar, +Both)
- 2 learning rates (0.001, 0.0003)

In [ ]:
# Define 8 model variations
variations = [
    # Base models
    {
        "name": "Base_LR0.001",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30"],
        "learning_rate": 0.001
    },
    {
        "name": "Base_LR0.0003",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30"],
        "learning_rate": 0.0003
    },
    
    # Base + Wind
    {
        "name": "Base+Wind_LR0.001",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_vs"],
        "learning_rate": 0.001
    },
    {
        "name": "Base+Wind_LR0.0003",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_vs"],
        "learning_rate": 0.0003
    },
    
    # Base + Solar
    {
        "name": "Base+Solar_LR0.001",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_srad"],
        "learning_rate": 0.001
    },
    {
        "name": "Base+Solar_LR0.0003",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_srad"],
        "learning_rate": 0.0003
    },
    
    # Base + Solar + Wind (Full)
    {
        "name": "Full_LR0.001",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_srad", "mean_vs"],
        "learning_rate": 0.001
    },
    {
        "name": "Full_LR0.0003",
        "var_list": ["mean_pr", "mean_tair", "mean_swe_lag_30", "mean_srad", "mean_vs"],
        "learning_rate": 0.0003
    },
]

print(f"✅ Defined {len(variations)} model variations:")
for i, v in enumerate(variations, 1):
    print(f"   {i}. {v['name']} - {len(v['var_list'])} features")

print(f"\n⏱️  Estimated time: {len(variations)} × 4-6 hours = ~32-48 hours total")
print(f"💰 Estimated cost: {len(variations)} × $3 = ~$24 total")

---
# Step 7: Load HUC Lists (10 seconds)

Load training and validation HUC lists

In [ ]:
# Load training HUCs
with open('exp1b_train_hucs.txt', 'r') as f:
    train_hucs = [line.strip() for line in f if line.strip()]

# Load validation HUCs
with open('exp1b_validation_hucs.txt', 'r') as f:
    val_hucs = [line.strip() for line in f if line.strip()]

print(f"✅ HUC lists loaded:")
print(f"   Training HUCs: {len(train_hucs)}")
print(f"   Validation HUCs: {len(val_hucs)}")
print(f"   Total: {len(train_hucs) + len(val_hucs)} HUCs")

# Show a few examples
print(f"\n   Example training HUCs: {train_hucs[:3]}")
print(f"   Example validation HUCs: {val_hucs[:3]}")

---
# Step 8: Load Data from S3 (3-5 minutes)

This downloads and preprocesses data for all HUCs from S3.

**This will take 3-5 minutes** - it's downloading 308 CSV files and computing normalization statistics.

☕ Good time for a coffee break!

In [ ]:
print("="*80)
print("LOADING DATA FROM S3")
print("="*80)

# We'll load data for the first variation, then reuse for others
var_list_max = variations[-1]['var_list']  # Use the fullest variable list

all_hucs = train_hucs + val_hucs

print(f"Loading {len(all_hucs)} HUCs from S3...")
print(f"Variables: {var_list_max}")
print("This may take 3-5 minutes...\n")

start_time = datetime.now()

df_dict, global_means, global_stds = LSTM_pp.pre_process(
    huc_list=all_hucs,
    var_list=var_list_max,
    UCLA=False,
    filter_dates=None,
    bucket_dict=sdc.create_bucket_dict("prod")
)

load_time = (datetime.now() - start_time).total_seconds()

print("\n" + "="*80)
print("✅ DATA LOADED SUCCESSFULLY!")
print("="*80)

print(f"\n📊 Loaded {len(df_dict)} HUCs in {load_time:.1f} seconds")
print(f"\n📈 Normalization stats:")
for var in var_list_max:
    print(f"   {var}: mean={global_means[var]:.3f}, std={global_stds[var]:.3f}")

# Split into train and val
df_dict_train = {huc: df_dict[huc] for huc in train_hucs}
df_dict_val = {huc: df_dict[huc] for huc in val_hucs}

print(f"\n✅ Split into:")
print(f"   Training: {len(df_dict_train)} HUCs")
print(f"   Validation: {len(df_dict_val)} HUCs")

# Show data shape for one HUC
sample_huc = train_hucs[0]
print(f"\n📋 Example data shape (HUC {sample_huc}):")
print(f"   Shape: {df_dict_train[sample_huc].shape}")
print(f"   Columns: {list(df_dict_train[sample_huc].columns)}")

---
# Step 9: Train All 8 Variations (4-6 hours EACH)

## ⚠️ IMPORTANT - READ BEFORE RUNNING:

### What Happens:
1. **Trains 8 models sequentially** (one after another)
2. **Each model takes 4-6 hours** (30 epochs)
3. **Total time: ~32-48 hours** (1.3-2 days)
4. **Saves checkpoints every 10 epochs** (automatic)
5. **Logs everything to MLflow** (automatic)

### Can You Close Your Laptop?
**YES!** Once you start this cell:
- Training runs on AWS server (not your laptop)
- You can close your laptop and walk away
- Reconnect anytime to check progress
- It will keep running 24/7 until all 8 models finish

### How to Monitor Progress:
1. **In this notebook:** Watch the output
2. **MLflow UI:** Run `!mlflow ui --backend-store-uri sqlite:///mlflow.db` in another cell
3. **Check later:** Just reconnect and scroll to see progress

### Cost Warning:
- This will cost ~$24 total ($0.53/hour × ~45 hours)
- Make sure you want to run ALL 8 variations
- Or modify the code below to run just 1-2 variations first

### To Run Just ONE Variation (Recommended First):
Change `variations` to `variations[:1]` below to test with just the first model.

---

**Ready to start? Run the cell below!**

In [ ]:
print("="*80)
print("STARTING FULL TRAINING - ALL 8 VARIATIONS")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n✅ You can now close your laptop!")
print(f"   Training will continue on AWS.")
print(f"   Reconnect anytime to check progress.\n")

# Setup MLflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Exp1B_Full_DeepSnowOnly")

# Store results
all_results = []

# TRAINING MODE - Choose what to run:
# variations[:1]  = Test with first variation only
# variations[1:]  = Skip first, run remaining 7
# variations      = Run all 8 variations
variations_to_run = variations  # ← Change this as needed

print(f"🎯 Training {len(variations_to_run)} out of {len(variations)} variations")
print(f"⏱️  Estimated time: {len(variations_to_run) * 4.5:.1f} hours\n")

# Train each variation
for idx, variation in enumerate(variations_to_run, 1):
    print("\n" + "="*80)
    print(f"VARIATION {idx}/{len(variations_to_run)}: {variation['name']}")
    print("="*80)
    print(f"   Variables: {variation['var_list']}")
    print(f"   Learning Rate: {variation['learning_rate']}")
    print(f"   Started: {datetime.now().strftime('%H:%M:%S')}")
    
    var_list = variation['var_list']
    learning_rate = variation['learning_rate']
    
    # Build params dict
    params = {
        'var_list': var_list,
        'device': device,
        'batch_size': batch_size,
        'seq_length': lookback,
        'lookback': lookback,
        'global_means': global_means,
        'global_stds': global_stds,
        'loss_type': 'mse',
        'learning_rate': learning_rate,
        'train_size_dimension': 'time',
        'train_size_fraction': 0.8,
        'num_workers': 0,
        'recursive_predict': False,
        'lag_swe_var_idx': var_list.index('mean_swe_lag_30')
    }
    
    # Initialize model
    input_size = len(var_list)
    model = LSTM_mod.SnowModel(
        input_size=input_size,
        hidden_size=hidden_size,
        num_class=1,
        num_layers=num_layers,
        dropout=dropout
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.MSELoss()
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"   ✅ Model initialized: {total_params:,} parameters")
    
    # Start MLflow run
    with mlflow.start_run(run_name=variation['name']):
        # Log parameters
        mlflow.log_params({
            "variation_name": variation['name'],
            "n_train_hucs": len(df_dict_train),
            "n_val_hucs": len(df_dict_val),
            "features": ",".join(var_list),
            "learning_rate": learning_rate,
            "n_epochs": n_epochs,
            "hidden_size": hidden_size,
            "device": str(device),
            "batch_size": batch_size,
            "lookback": lookback
        })
        
        print(f"   ✅ MLflow run started\n")
        
        # Training loop
        variation_start = datetime.now()
        
        for epoch in range(n_epochs):
            epoch_start = datetime.now()
            print(f"\n   ━━━ Epoch {epoch + 1}/{n_epochs} ━━━")
            
            # Train
            print(f"      🏋️  Training on {len(df_dict_train)} HUCs...")
            train_loss = LSTM_tr.pre_train(
                model=model,
                optimizer=optimizer,
                loss_fn=loss_fn,
                df_dict=df_dict_train,
                params=params,
                epoch=epoch
            )
            
            # Validate
            print(f"      🔍 Validating on {len(df_dict_val)} HUCs...")
            val_metrics = LSTM_tr.evaluate(
                model_dawgs=model,
                df_dict=df_dict_val,
                params=params,
                epoch=epoch
            )
            
            # Extract metrics
            val_kge = val_metrics[1].get('test_kge', 0)
            print(f"      📊 Validation KGE: {val_kge:.4f}")
            
            # Log to MLflow
            if train_loss is not None:
                mlflow.log_metrics({"train_loss": train_loss}, step=epoch)
            mlflow.log_metrics({"val_kge": val_kge}, step=epoch)
            
            # Save checkpoint every 10 epochs
            if (epoch + 1) % 10 == 0 or epoch == n_epochs - 1:
                LSTM_tr.save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    epoch=epoch,
                    metrics={'val_kge': val_kge},
                    filepath=f"checkpoint_{variation['name']}_epoch_{epoch}.pt",
                    params=params
                )
                print(f"      💾 Checkpoint saved")
            
            epoch_duration = (datetime.now() - epoch_start).total_seconds()
            print(f"      ⏱️  Epoch time: {epoch_duration:.1f}s")
        
        # ===================================================================
        # SAVE FINAL MODEL TO MLFLOW
        # ===================================================================
        print(f"\n   💾 Saving final model to MLflow...")
        
        # Save model to MLflow model registry
        mlflow.pytorch.log_model(
            model, 
            artifact_path="model",
            registered_model_name=f"Exp1B_{variation['name']}"
        )
        
        # Save final checkpoint with clear name
        final_checkpoint_path = f"checkpoint_{variation['name']}_FINAL.pt"
        LSTM_tr.save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=n_epochs - 1,
            metrics={'final_val_kge': val_kge},
            filepath=final_checkpoint_path,
            params=params
        )
        
        print(f"   ✅ Model saved!")
        print(f"      - MLflow registry: Exp1B_{variation['name']}")
        print(f"      - Final checkpoint: {final_checkpoint_path}")
        # ===================================================================
        
        # Variation complete
        variation_duration = (datetime.now() - variation_start).total_seconds() / 3600
        print(f"\n   ✅ {variation['name']} COMPLETE!")
        print(f"   ⏱️  Duration: {variation_duration:.2f} hours")
        print(f"   📊 Final KGE: {val_kge:.4f}")
        print(f"   📍 Finished: {datetime.now().strftime('%H:%M:%S')}")
        
        # Store result
        all_results.append({
            'name': variation['name'],
            'final_val_kge': val_kge,
            'duration_hours': variation_duration
        })

# All training complete!
print("\n" + "="*80)
print("🎉 ALL TRAINING COMPLETE!")
print("="*80)
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Show summary
print("\n📊 Results Summary:")
results_df = pd.DataFrame(all_results)
print(results_df.to_string(index=False))

print(f"\n🏆 Best model: {results_df.loc[results_df['final_val_kge'].idxmax(), 'name']}")
print(f"   Best KGE: {results_df['final_val_kge'].max():.4f}")

# Save results
results_df.to_csv('exp1b_training_results.csv', index=False)
print("\n💾 Results saved to: exp1b_training_results.csv")

print("\n" + "="*80)
print("📦 What's Been Saved:")
print("="*80)
print("✅ MLflow Database: mlflow.db")
print("   - All metrics, parameters, and model references")
print("✅ Model Registry: 8 models registered in MLflow")
print("   - Load with: mlflow.pytorch.load_model('models:/Exp1B_[name]/latest')")
print("✅ Checkpoint Files: checkpoint_*_FINAL.pt")
print("   - One for each variation")
print("✅ Results CSV: exp1b_training_results.csv")
print("\n⚠️  IMPORTANT: Stop instance when done!")
print("   aws sagemaker stop-notebook-instance --notebook-instance-name st-gnn-T4x1 --region us-west-2")

---
# Step 10: View Results in MLflow (Optional)

Run this to start MLflow UI and view all experiment results

In [ ]:
# Start MLflow UI
print("Starting MLflow UI...")
print("\nAccess at: http://localhost:5000")
print("\nPress Ctrl+C to stop")

!mlflow ui --backend-store-uri sqlite:///mlflow.db

---
# 🎉 Training Complete!

## What You Have Now:

✅ **8 trained models** (or however many you ran)
✅ **MLflow logs** with all metrics and parameters
✅ **Checkpoints** saved every 10 epochs
✅ **Results CSV** with summary statistics

---

## Next Steps:

1. **Review results** in MLflow UI
2. **Identify best model** (highest validation KGE)
3. **Test on Test Set A** (78 random HUCs)
4. **Test on Test Set B** (81 Yakima/Naches HUCs)
5. **Compare to published results**

---

## ⚠️ IMPORTANT: Stop the Instance!

When you're done, **STOP the notebook instance** to avoid charges:

```bash
aws sagemaker stop-notebook-instance \
  --notebook-instance-name st-gnn-T4x1 \
  --region us-west-2
```

Or it will keep charging $0.53/hour!

---

**Great work! 🚀**